# 예제 franka_ex14: FR3 YZ 원형 경로 — 실시간 IK 스트리밍

`ex13` 과 같은 원이지만 **실시간(30Hz)** 으로 매 주기마다:
1. 원 위의 다음 목표점 계산
2. MoveIt `compute_ik` 로 목표 관절각 계산 (현재 관절을 시드로)
3. `JointTrajectory` 메시지를 `fr3_arm_controller` 에 직접 발행

**6-DOF 예제와 다른 점**
- 컨트롤러 토픽 이름: 6-DOF 의 `/arm_controller/joint_trajectory` → FR3 `/fr3_arm_controller/joint_trajectory`
- 조인트 7개 (`fr3_joint1..7`)
- 원 반경 15cm (FR3 스케일), 중심 `(0.55, 0.20, 0.55)`
- planning group `fr3_arm`, frame `fr3_link0`, EE `fr3_hand_tcp`
- `use_sim_time=True`

**ex13 vs ex14**
- ex13: 오프라인 `compute_cartesian_path` → 한 번에 trajectory 받아 `execute_trajectory` (오픈루프 재생)
- ex14: 매 주기마다 IK 풀고 `JointTrajectory` 한 점씩 발행 (실시간 폐루프 — 다만 보정은 최소)

**주의** — Servo 노드는 안 쓴다. Servo 의 특이점 감지가 과보수적이어서 `ex11` 처럼 자주 정지하는 문제가 있어서다.
대신 `compute_ik` 만 사용하므로 IK 해가 있는 한 동작 (단, 컨트롤러 안정성은 PD gain 에 의존).

## 실행 절차

이 노트북은 별도로 띄운 MoveIt + RViz 의 `move_group` 액션 서버에 클라이언트로 붙는다.

> ⚠ 다른 로봇용 MoveIt launch 가 떠 있으면 같은 토픽으로 충돌할 수 있다.
> 시작 전에 `pgrep -af 'ros2 launch'` 로 잔존 프로세스가 없는지 확인하자.

### 터미널 1 — Franka FR3 (Gazebo Sim) + MoveIt + RViz 기동

```bash
source /opt/ros/jazzy/setup.bash
source ~/robot_arm/install/setup.bash
ros2 launch franka_tutorials franka_gazebo_moveit.launch.py
```

RViz 가 뜨면 **`MarkerArray` Display 를 추가하고 Topic 을 `/path_markers` 로 설정**한다.
Fixed Frame 은 `fr3_link0` 로 둔다.

### 터미널 2 — Jupyter 기동

```bash
source ~/venv/ros_jazzy/bin/activate
source /opt/ros/jazzy/setup.bash
source ~/robot_arm/install/setup.bash
cd ~/robot_arm/src/robotarm_tutorials/robot_arm_tutorials/robot_arm_tutorials
jupyter lab
```

셀을 위에서 아래로 순서대로 실행한다 (`Shift+Enter`).

## 1. 상수

In [ ]:
PLANNING_GROUP    = 'fr3_arm'
REFERENCE_FRAME   = 'fr3_link0'
END_EFFECTOR_LINK = 'fr3_hand_tcp'
ARM_JOINTS        = ['fr3_joint1', 'fr3_joint2', 'fr3_joint3',
                     'fr3_joint4', 'fr3_joint5', 'fr3_joint6',
                     'fr3_joint7']
MARKER_TOPIC = '/path_markers'

ARM_TRAJ_TOPIC = '/fr3_arm_controller/joint_trajectory'

RADIUS         = 0.15
NUM_LOOPS      = 3
CONTROL_RATE   = 30.0   # Hz
ANGULAR_SPEED  = 0.5    # rad/s — 원 위 진행 속도
CIRCLE_CENTER  = (0.55, 0.20, 0.55)

## 2. 초기화

In [ ]:
import rclpy, math, time, threading
from rclpy.node import Node
from rclpy.action import ActionClient
from rclpy.parameter import Parameter
from sensor_msgs.msg import JointState
from moveit_msgs.action import MoveGroup
from moveit_msgs.srv import GetPositionIK
from moveit_msgs.msg import RobotState, MoveItErrorCodes
from trajectory_msgs.msg import JointTrajectory, JointTrajectoryPoint
from visualization_msgs.msg import MarkerArray, Marker
from std_msgs.msg import ColorRGBA
from geometry_msgs.msg import Pose, Point
from builtin_interfaces.msg import Duration
import tf2_ros

In [ ]:
try:
    rclpy.init()
except RuntimeError:
    pass  # 이미 초기화된 경우 무시

In [ ]:
node = Node(
    'franka_ex14_circle_servo_demo',
    parameter_overrides=[Parameter('use_sim_time', value=True)],
)
move_client = ActionClient(node, MoveGroup, 'move_action')

joint_state = {'msg': None}
node.create_subscription(
    JointState, 'joint_states',
    lambda msg: joint_state.update(msg=msg), 10,
)
node.get_logger().info('=== franka_ex14 노트북 노드 생성 완료 ===')
marker_pub = node.create_publisher(MarkerArray, MARKER_TOPIC, 10)
ik_client = node.create_client(GetPositionIK, 'compute_ik')
traj_pub = node.create_publisher(JointTrajectory, ARM_TRAJ_TOPIC, 10)
tf_buffer = tf2_ros.Buffer()
tf_listener = tf2_ros.TransformListener(tf_buffer, node)

## 3. 서버 준비

In [ ]:
import time

def wait_for_ready(timeout_sec: float = 30.0) -> None:
    if not move_client.wait_for_server(timeout_sec=timeout_sec):
        raise RuntimeError('MoveGroup 액션 서버 연결 실패')
    start = time.time()
    while joint_state['msg'] is None:
        rclpy.spin_once(node, timeout_sec=0.1)
        if time.time() - start > timeout_sec:
            raise RuntimeError('joint_states 수신 실패')
    node.get_logger().info('action server + /joint_states 준비됨')

wait_for_ready()

## 4. SRDF `ready` 자세

In [ ]:
from rclpy.parameter_client import AsyncParameterClient
import xml.etree.ElementTree as ET

def fetch_srdf_xml(timeout_sec: float = 10.0) -> str:
    client = AsyncParameterClient(node, 'move_group')
    if not client.wait_for_services(timeout_sec=timeout_sec):
        raise RuntimeError('move_group 파라미터 서비스 연결 실패')
    future = client.get_parameters(['robot_description_semantic'])
    rclpy.spin_until_future_complete(node, future, timeout_sec=timeout_sec)
    return future.result().values[0].string_value

def parse_named_pose(srdf_xml: str, name: str, group: str) -> dict:
    root = ET.fromstring(srdf_xml)
    for gs in root.findall('group_state'):
        if gs.attrib.get('group') == group and gs.attrib.get('name') == name:
            return {j.attrib['name']: float(j.attrib.get('value', '0'))
                    for j in gs.findall('joint')}
    raise RuntimeError(f'SRDF group_state "{name}" (group={group}) 없음')

def load_named_pose(name: str, timeout_sec: float = 10.0) -> dict:
    return parse_named_pose(fetch_srdf_xml(timeout_sec), name, PLANNING_GROUP)

ready_target = load_named_pose('ready')
node.get_logger().info(f'ready: {ready_target}')

## 5. Pose / MoveGroup 헬퍼

In [ ]:
import math
import tf_transformations
from geometry_msgs.msg import Pose, Point, Quaternion

def euler_to_quaternion(roll: float, pitch: float, yaw: float) -> Quaternion:
    q = tf_transformations.quaternion_from_euler(roll, pitch, yaw)
    return Quaternion(x=q[0], y=q[1], z=q[2], w=q[3])

def make_pose(x: float, y: float, z: float,
              roll: float = 0.0, pitch: float = 0.0, yaw: float = 0.0) -> Pose:
    pose = Pose()
    pose.position = Point(x=x, y=y, z=z)
    pose.orientation = euler_to_quaternion(roll, pitch, yaw)
    return pose

In [ ]:
from moveit_msgs.msg import (
    Constraints, JointConstraint,
    PositionConstraint, OrientationConstraint, BoundingVolume,
    MotionPlanRequest, PlanningOptions, MoveItErrorCodes,
)
from shape_msgs.msg import SolidPrimitive
from geometry_msgs.msg import Vector3

def make_joint_constraints(joint_values: dict, tol: float = 0.01) -> Constraints:
    c = Constraints()
    for jname, val in joint_values.items():
        c.joint_constraints.append(JointConstraint(
            joint_name=jname, position=val,
            tolerance_above=tol, tolerance_below=tol, weight=1.0,
        ))
    return c

def make_position_constraint(pose: Pose, tol: float = 0.01) -> PositionConstraint:
    pc = PositionConstraint()
    pc.header.frame_id = REFERENCE_FRAME
    pc.link_name = END_EFFECTOR_LINK
    pc.target_point_offset = Vector3(x=0.0, y=0.0, z=0.0)
    bv = BoundingVolume()
    sphere = SolidPrimitive()
    sphere.type = SolidPrimitive.SPHERE
    sphere.dimensions = [tol]
    bv.primitives.append(sphere)
    sp = Pose()
    sp.position = Point(x=pose.position.x, y=pose.position.y, z=pose.position.z)
    sp.orientation.w = 1.0
    bv.primitive_poses.append(sp)
    pc.constraint_region = bv
    pc.weight = 1.0
    return pc

def make_orientation_constraint(pose_or_quat, tol: float = 0.01) -> OrientationConstraint:
    oc = OrientationConstraint()
    oc.header.frame_id = REFERENCE_FRAME
    oc.link_name = END_EFFECTOR_LINK
    if hasattr(pose_or_quat, 'orientation'):
        oc.orientation = pose_or_quat.orientation
    else:
        oc.orientation = pose_or_quat
    oc.absolute_x_axis_tolerance = tol
    oc.absolute_y_axis_tolerance = tol
    oc.absolute_z_axis_tolerance = tol
    oc.weight = 1.0
    return oc

def make_plan_request(vel: float = 0.3, acc: float = 0.3,
                      attempts: int = 5, plan_time: float = 10.0,
                      planner_id: str = '') -> MotionPlanRequest:
    req = MotionPlanRequest()
    req.group_name = PLANNING_GROUP
    req.num_planning_attempts = attempts
    req.allowed_planning_time = plan_time
    req.max_velocity_scaling_factor = vel
    req.max_acceleration_scaling_factor = acc
    if planner_id:
        req.planner_id = planner_id
    return req

def send_move_goal(req: MotionPlanRequest, plan_only: bool = False):
    goal = MoveGroup.Goal()
    goal.request = req
    goal.planning_options = PlanningOptions(
        plan_only=plan_only, replan=not plan_only, replan_attempts=3 if not plan_only else 0)
    sf = move_client.send_goal_async(goal)
    rclpy.spin_until_future_complete(node, sf)
    handle = sf.result()
    if handle is None or not handle.accepted:
        return MoveItErrorCodes.PLANNING_FAILED, None
    rf = handle.get_result_async()
    rclpy.spin_until_future_complete(node, rf)
    res = rf.result().result
    return res.error_code.val, res.planned_trajectory

def go_to_joint_goal(joint_values: dict, vel: float = 0.3, acc: float = 0.3) -> bool:
    req = make_plan_request(vel, acc)
    req.goal_constraints.append(make_joint_constraints(joint_values))
    code_val, _ = send_move_goal(req, plan_only=False)
    ok = (code_val == MoveItErrorCodes.SUCCESS)
    if not ok:
        node.get_logger().error(f'joint goal 실패 error_code={code_val}')
    return ok

def go_to_pose_goal(pose: Pose, vel: float = 0.3, acc: float = 0.3) -> bool:
    req = make_plan_request(vel, acc)
    c = Constraints()
    c.position_constraints.append(make_position_constraint(pose))
    c.orientation_constraints.append(make_orientation_constraint(pose))
    req.goal_constraints.append(c)
    code_val, _ = send_move_goal(req, plan_only=False)
    ok = (code_val == MoveItErrorCodes.SUCCESS)
    if not ok:
        node.get_logger().error(f'pose goal 실패 error_code={code_val} (IK 해 없음 가능)')
    return ok

def plan_to_joint_goal(joint_values: dict, vel: float = 0.3, acc: float = 0.3,
                       planner_id: str = '', plan_time: float = 10.0):
    req = make_plan_request(vel, acc, plan_time=plan_time, planner_id=planner_id)
    req.goal_constraints.append(make_joint_constraints(joint_values))
    code_val, traj = send_move_goal(req, plan_only=True)
    return code_val == MoveItErrorCodes.SUCCESS, traj

def plan_to_pose_goal(pose: Pose, vel: float = 0.3, acc: float = 0.3,
                      planner_id: str = '', plan_time: float = 10.0):
    req = make_plan_request(vel, acc, plan_time=plan_time, planner_id=planner_id)
    c = Constraints()
    c.position_constraints.append(make_position_constraint(pose))
    c.orientation_constraints.append(make_orientation_constraint(pose))
    req.goal_constraints.append(c)
    code_val, traj = send_move_goal(req, plan_only=True)
    return code_val == MoveItErrorCodes.SUCCESS, traj

## 6. 실시간 IK 헬퍼

매 주기마다 호출. 현재 조인트를 시드로 IK 해 1개 받아 `fr3_arm_controller` 로 직접 발행.

In [ ]:
def get_current_joints() -> dict:
    if joint_state['msg'] is None:
        return {}
    msg = joint_state['msg']
    return {n: p for n, p in zip(msg.name, msg.position) if n in ARM_JOINTS}

def compute_ik(target_pose: Pose, timeout_sec: float = 0.05):
    req = GetPositionIK.Request()
    req.ik_request.group_name = PLANNING_GROUP
    req.ik_request.pose_stamped.header.frame_id = REFERENCE_FRAME
    req.ik_request.pose_stamped.pose = target_pose
    req.ik_request.timeout.sec = 0
    req.ik_request.timeout.nanosec = int(timeout_sec * 1e9)
    cur = get_current_joints()
    if cur:
        rs = RobotState()
        rs.joint_state.name = list(cur.keys())
        rs.joint_state.position = list(cur.values())
        req.ik_request.robot_state = rs
    fut = ik_client.call_async(req)
    deadline = time.time() + 0.1
    while not fut.done() and time.time() < deadline:
        rclpy.spin_once(node, timeout_sec=0.005)
    if fut.done() and fut.result() is not None:
        res = fut.result()
        if res.error_code.val == MoveItErrorCodes.SUCCESS:
            return {n: p for n, p in zip(res.solution.joint_state.name,
                                          res.solution.joint_state.position)
                    if n in ARM_JOINTS}
    return None

def publish_joint_cmd(joint_values: dict, duration_s: float = 0.05):
    msg = JointTrajectory()
    msg.header.stamp = node.get_clock().now().to_msg()
    msg.joint_names = ARM_JOINTS
    pt = JointTrajectoryPoint()
    pt.positions = [joint_values.get(n, 0.0) for n in ARM_JOINTS]
    sec = int(duration_s)
    pt.time_from_start = Duration(sec=sec, nanosec=int((duration_s - sec) * 1e9))
    msg.points.append(pt)
    traj_pub.publish(msg)

## 7. 마커 — 목표 원 + 실제 경로

In [ ]:
COLOR_TARGET = ColorRGBA(r=0.1, g=0.8, b=0.1, a=1.0)
COLOR_ACTUAL = ColorRGBA(r=1.0, g=0.4, b=0.0, a=0.9)

def publish_target_circle(cx, cy, cz, r):
    stamp = node.get_clock().now().to_msg()
    line = Marker()
    line.header.frame_id = REFERENCE_FRAME
    line.header.stamp = stamp
    line.ns = 'target_circle'
    line.id = 0
    line.type = Marker.LINE_STRIP
    line.action = Marker.ADD
    line.scale.x = 0.005
    line.color = COLOR_TARGET
    line.pose.orientation.w = 1.0
    for i in range(49):
        th = 2.0 * math.pi * i / 48
        line.points.append(Point(x=cx, y=cy + r * math.sin(th), z=cz + r * math.cos(th)))
    marker_pub.publish(MarkerArray(markers=[line]))

def publish_actual_path(pts):
    if len(pts) < 2:
        return
    stamp = node.get_clock().now().to_msg()
    line = Marker()
    line.header.frame_id = REFERENCE_FRAME
    line.header.stamp = stamp
    line.ns = 'actual_line'
    line.id = 0
    line.type = Marker.LINE_STRIP
    line.action = Marker.ADD
    line.scale.x = 0.004
    line.color = COLOR_ACTUAL
    line.pose.orientation.w = 1.0
    line.points = [Point(x=p[0], y=p[1], z=p[2]) for p in pts]
    marker_pub.publish(MarkerArray(markers=[line]))

## 8. 시나리오 — ready → 중심 → 시작점 → 실시간 IK 루프

In [ ]:
# 8-1. ready
go_to_joint_goal(ready_target, vel=0.4)
time.sleep(0.5)

# 8-2. 원 중심
cx, cy, cz = CIRCLE_CENTER
gripper_q = euler_to_quaternion(0.0, math.pi / 2, 0.0)
center_pose = Pose()
center_pose.position = Point(x=cx, y=cy, z=cz)
center_pose.orientation = gripper_q
node.get_logger().info(f'--- 중심 ({cx}, {cy}, {cz}) 이동 ---')
ok = go_to_pose_goal(center_pose, vel=0.3)
if not ok:
    raise RuntimeError('중심 이동 실패 — 좌표를 reach 안쪽으로')
time.sleep(0.5)

# 8-3. 시작점 (z+R)
start_pose = Pose()
start_pose.position = Point(x=cx, y=cy, z=cz + RADIUS)
start_pose.orientation = gripper_q
ok = go_to_pose_goal(start_pose, vel=0.3)
if not ok:
    raise RuntimeError('시작점 이동 실패')
time.sleep(0.5)

publish_target_circle(cx, cy, cz, RADIUS)

# 8-4. 카운트다운
for i in [3, 2, 1]:
    node.get_logger().info(f'  >>> {i}')
    time.sleep(1.0)
node.get_logger().info('  >>> START!')

# 8-5. 실시간 IK 루프
dt = 1.0 / CONTROL_RATE
theta = 0.0
total_angle = 2.0 * math.pi * NUM_LOOPS
ik_fail = 0
errors = []
actual_pts = []

start_t = time.time()
while theta < total_angle:
    loop_t = time.time()
    theta += ANGULAR_SPEED * dt
    tgt_y = cy + RADIUS * math.sin(theta)
    tgt_z = cz + RADIUS * math.cos(theta)
    target = Pose()
    target.position = Point(x=cx, y=tgt_y, z=tgt_z)
    target.orientation = gripper_q

    ik_sol = compute_ik(target)
    if ik_sol:
        publish_joint_cmd(ik_sol, duration_s=dt * 1.5)
        ik_fail = 0
    else:
        ik_fail += 1
        if ik_fail % 30 == 1:
            node.get_logger().warn(f'IK 실패 (θ={math.degrees(theta):.0f}°)')

    # 현재 EE 측정
    try:
        tr = tf_buffer.lookup_transform(REFERENCE_FRAME, END_EFFECTOR_LINK, rclpy.time.Time())
        cur = (tr.transform.translation.x,
               tr.transform.translation.y,
               tr.transform.translation.z)
        actual_pts.append(cur)
        err = math.sqrt((cur[0] - cx)**2 + (cur[1] - tgt_y)**2 + (cur[2] - tgt_z)**2)
        errors.append(err)
    except Exception:
        pass

    # 주기 유지
    elapsed = time.time() - loop_t
    if dt - elapsed > 0:
        time.sleep(dt - elapsed)

publish_actual_path(actual_pts)
total = time.time() - start_t
node.get_logger().info(f'실시간 IK 완료: {total:.1f}s, 추적 샘플 {len(actual_pts)}점')
if errors:
    import statistics
    node.get_logger().info(
        f'트래킹 에러: max={max(errors)*100:.2f}cm, '
        f'rms={math.sqrt(sum(e*e for e in errors)/len(errors))*100:.2f}cm'
    )

## 9. ready 복귀

In [ ]:
go_to_joint_goal(ready_target, vel=0.4)
node.get_logger().info('=== franka_ex14 완료! ===')

## 10. 정리

In [ ]:
node.destroy_node()
try:
    rclpy.shutdown()
except Exception:
    pass